In [ ]:
import pandas as pd
import glob

# 获取所有分批输出的文件
files = sorted(glob.glob("phewas_dataset_part*.csv"))

# 依次读取并合并
dfs = [pd.read_csv(f) for f in files]

# 按 eid 列合并（如果每个文件都包含 eid 且顺序一致）
from functools import reduce
df_merged = reduce(lambda left, right: pd.merge(left, right, on='participant.eid'), dfs)
df_merged

In [ ]:
df_merged[df_merged.columns[1:]]

In [ ]:
df_merged.to_csv('mri_dataset.csv', index=False)

In [ ]:
import pandas as pd
df_merged = pd.read_csv('mri_dataset.csv')
df_merged

In [ ]:
df_merged = df_merged[df_merged.iloc[:, 1:].notna().any(axis=1)]
df_merged

In [ ]:
code = pd.read_csv(r"depression_mri\code.csv")
code

In [ ]:
# 从 df_merged（排除第一列）中，提取列名包含 code.csv 中 Field ID 任意值的列（不区分大小写）
fields = code['Field ID'].dropna().astype(str).str.lower().str.strip().unique().tolist()
first_col = df_merged.columns[0]

# 匹配：如果任一 Field ID 子串出现在列名中，则保留该列
code_cols = [c for c in df_merged.columns[1:] if any(f in c.lower() for f in fields)]

# 对结果做简单检查并提取
print(f"匹配到 {len(code_cols)} 列，例如：{code_cols[:10]}")
df_code = df_merged[[first_col] + code_cols]
df = df_code.T.drop_duplicates().T
df

In [ ]:
cols = df.columns.tolist()
for col in cols:
    if col.endswith('_x'):
        base = col[:-2]
        col_x = col
        if col_x in df.columns:
            df.rename(columns={col_x: base}, inplace=True)
df

In [ ]:
ss = ['participant.eid'] + df.columns.tolist()[1:9] + df.columns.tolist()[207:231] #皮质下
cs = ['participant.eid'] + [c for c in df.columns[1:] if c not in ss] #皮层

In [ ]:
df_cs = df[cs]
df_cs = pd.merge(df_cs, df_merged[['participant.eid',
                                   'participant.p26721_i2_x', 'participant.p26755_i2_x',
                                   'participant.p26822_i2_x', 'participant.p26856_i2_x']], on='participant.eid')
df_cs['ALeft_temporalpole'] = (df_cs['participant.p26721_i2_x'] - df_cs.iloc[:, 1:34].sum(axis=1)).round(1).clip(lower=0)
df_cs['MLeft_temporalpole'] = (df_cs['participant.p26755_i2_x']*34 - df_cs.iloc[:, 34:67].sum(axis=1)).round(5).clip(lower=0)
df_cs['VLeft_temporalpole'] = (df_cs['MLeft_temporalpole'] * df_cs['ALeft_temporalpole']).round(1).clip(lower=0)

df_cs['ARight_temporalpole'] = (df_cs['participant.p26822_i2_x'] - df_cs.iloc[:, 100:133].sum(axis=1)).round(1).clip(lower=0)
df_cs['MRight_temporalpole'] = (df_cs['participant.p26856_i2_x']*34 - df_cs.iloc[:, 133:166].sum(axis=1)).round(5).clip(lower=0)
df_cs['VRight_temporalpole'] = (df_cs['MRight_temporalpole'] * df_cs['ARight_temporalpole']).round(1).clip(lower=0)
df_cs
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib
import numpy as np

pro_f_lst = df_cs.columns.tolist()[1:]
scaler = StandardScaler()
scaler.fit(df_cs[pro_f_lst])
tmp = scaler.transform(df_cs[pro_f_lst])
df_prot_train_tissue = pd.DataFrame(tmp, index=df_cs[pro_f_lst].index, columns=df_cs[pro_f_lst].columns)
other_cols = [col for col in df_cs.columns if col not in pro_f_lst]
df_cs = pd.concat([ df_cs[other_cols],df_prot_train_tissue], axis=1)            
joblib.dump(scaler, 'model/cs_zscore_scaler.pkl',compress=3)

# 设定一个阈值，唯一值数量少于这个值的数字列被视为分类变量
max_unique_for_categorical = 20

numeric_cols = []
categorical_cols = []

for col in df_cs.columns[1:]:
    if df_cs[col].dtype in [np.int64, np.float64]:
        # 数字列：检查唯一值数量
        if df_cs[col].nunique() <= max_unique_for_categorical:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)
    else:
        # 非数字列直接视为分类变量
        categorical_cols.append(col)
if len(numeric_cols) > 0:
    num_imputer = SimpleImputer(strategy='most_frequent')
    df_cs[numeric_cols] = num_imputer.fit_transform(df_cs[numeric_cols])
    
if len(categorical_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_cs[categorical_cols] = cat_imputer.fit_transform(df_cs[categorical_cols])
df_cs

In [ ]:
df_cs.to_csv('mri_cs_dataset.csv', index=False)

In [ ]:
# weighted_sum = 0
# for i in range(100, 133):  # 遍历厚度列
#     area_col = df_cs.iloc[:, i]  
#     thickness_col = df_cs.iloc[:, i+33]    # 对应的第i+33列面积
#     weighted_sum += thickness_col * area_col
# df_cs['VRight_temporalpole'] = (df_cs['participant.p26856_i2_x'] * df_cs['participant.p26822_i2_x'] - weighted_sum).round(1).clip(lower=0)
# df_cs['MRight_temporalpole'] = (df_cs['VRight_temporalpole'] / df_cs['ARight_temporalpole']).where(df_cs['ARight_temporalpole'] != 0, 0).round(5).clip(lower=0)

# weighted_sum = 0
# for i in range(1, 34):  # 遍历厚度列
#     area_col = df_cs.iloc[:, i]  
#     thickness_col = df_cs.iloc[:, i+33]    # 对应的第i+33列面积
#     weighted_sum += thickness_col * area_col
# df_cs['MLeft_temporalpole'] = ((df_cs['participant.p26721_i2_x'] * df_cs['participant.p26755_i2_x'] - weighted_sum) / 
#                                (df_cs['participant.p26721_i2_x'] - df_cs.iloc[:, 1:34].sum(axis=1))).round(5).clip(lower=0)

In [ ]:
df_ss = df[ss]
pro_f_lst = df_ss.columns.tolist()[1:]
scaler = StandardScaler()
scaler.fit(df_ss[pro_f_lst])
tmp = scaler.transform(df_ss[pro_f_lst])
df_prot_train_tissue = pd.DataFrame(tmp, index=df_ss[pro_f_lst].index, columns=df_ss[pro_f_lst].columns)
other_cols = [col for col in df_ss.columns if col not in pro_f_lst]
df_ss = pd.concat([df_ss[other_cols],df_prot_train_tissue], axis=1)            
joblib.dump(scaler, 'model/ss_zscore_scaler.pkl',compress=3)

# 设定一个阈值，唯一值数量少于这个值的数字列被视为分类变量
max_unique_for_categorical = 20

numeric_cols = []
categorical_cols = []

for col in df_ss.columns[1:]:
    if df_ss[col].dtype in [np.int64, np.float64]:
        # 数字列：检查唯一值数量
        if df_ss[col].nunique() <= max_unique_for_categorical:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)
    else:
        # 非数字列直接视为分类变量
        categorical_cols.append(col)
if len(numeric_cols) > 0:
    num_imputer = SimpleImputer(strategy='most_frequent')
    df_ss[numeric_cols] = num_imputer.fit_transform(df_ss[numeric_cols])
    
if len(categorical_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_ss[categorical_cols] = cat_imputer.fit_transform(df_ss[categorical_cols])
df_ss

In [ ]:
df_ss.to_csv('mri_ss_dataset.csv', index=False)

In [ ]:
code_cols = [c for c in df_merged.columns[1:] if any(f in c.lower() for f in ['26517','26518','26528'])]
first_col = df_merged.columns[0]
df_gv = df_merged[[first_col] + code_cols]
df_gv['CGV'] = df_gv['participant.p26518_i2'] - df_gv['participant.p26517_i2']
pro_f_lst = df_gv.columns.tolist()[1:]
scaler = StandardScaler()
scaler.fit(df_gv[pro_f_lst])
tmp = scaler.transform(df_gv[pro_f_lst])
df_prot_train_tissue = pd.DataFrame(tmp, index=df_gv[pro_f_lst].index, columns=df_gv[pro_f_lst].columns)
other_cols = [col for col in df_gv.columns if col not in pro_f_lst]
df_gv = pd.concat([df_gv[other_cols], df_prot_train_tissue], axis=1)            
joblib.dump(scaler, 'model/gv_zscore_scaler.pkl',compress=3)

# 设定一个阈值，唯一值数量少于这个值的数字列被视为分类变量
max_unique_for_categorical = 20

numeric_cols = []
categorical_cols = []

for col in df_gv.columns[1:]:
    if df_gv[col].dtype in [np.int64, np.float64]:
        # 数字列：检查唯一值数量
        if df_gv[col].nunique() <= max_unique_for_categorical:
            categorical_cols.append(col)
        else:
            numeric_cols.append(col)
    else:
        # 非数字列直接视为分类变量
        categorical_cols.append(col)
if len(numeric_cols) > 0:
    num_imputer = SimpleImputer(strategy='most_frequent')
    df_gv[numeric_cols] = num_imputer.fit_transform(df_gv[numeric_cols])
    
if len(categorical_cols) > 0:
    cat_imputer = SimpleImputer(strategy='most_frequent')
    df_gv[categorical_cols] = cat_imputer.fit_transform(df_gv[categorical_cols])
df_gv

In [ ]:
df_gv.to_csv('mri_gv_dataset.csv', index=False)

In [ ]:
# imp_df = pd.read_csv(r"depression_protein\Top_90_InfoGain_features.csv", usecols = ['Analytes', 'Ensemble_cv'])
# imp_df.rename(columns = {'Ensemble_cv': 'pro_imp'}, inplace = True)
# AUC_df = pd.read_csv(r'depression_protein\Delong_Selection_Results.csv')
# mydf = pd.merge(AUC_df, imp_df, how = 'left', on = ['Analytes'])
# mydf_top15 = mydf[:30]
myout_df= pd.read_csv(r"Dep_Cox_improved4.csv")
# Filter proteins with significant FDR-corrected p-values
significant_proteins = myout_df[myout_df['p_val_bfi'] < 0.05]
significant_proteins = significant_proteins.sort_values(by='p_val_bfi')
sig_proteins_list = significant_proteins['Pro_code'].tolist()
protein_data = pd.read_csv(r"significant_proteins_data_processed1.csv")
#feature_cols = mydf_top15['Analytes'].tolist()
data_protein = protein_data[['Participant ID','depressed-after','Region_Code'] +sig_proteins_list]
data_protein 

In [ ]:
data_protein.to_csv('protein_data.csv', index=False)

In [ ]:
# imp_df = pd.read_csv(r"depression_nmr\Top_90_InfoGain_features.csv", usecols = ['Analytes', 'Ensemble_cv'])
# imp_df.rename(columns = {'Ensemble_cv': 'pro_imp'}, inplace = True)
# AUC_df = pd.read_csv(r'depression_nmr\Delong_Selection_Results.csv')
# mydf = pd.merge(AUC_df, imp_df, how = 'left', on = ['Analytes'])
# mydf_top15 = mydf[:30]
import pandas as pd
bio_data = pd.read_csv(r"significant_nmr_data_processed1.csv")
feature_cols = bio_data.columns.difference(['Participant ID','depressed-after','Region_Code','anxiety','any','any_before','bd','dementia','depressed',  
'menta_after','menta_before','mental','mental_before','scz','sd','sleep','sud']).tolist()
data_bio = bio_data[['Participant ID','any','Region_Code'] +feature_cols]
data_bio 

In [ ]:
data_bio.to_csv('bio_data.csv', index=False)

In [ ]:
import re
phe_data = pd.read_csv(r"phedata_merged_depression_cor.csv",index_col=0)
icd = pd.read_csv(r"prediction_results.csv")
icd.rename(columns={'patient_id':'Participant ID'}, inplace=True)
data = pd.merge(phe_data, icd[['Participant ID', 'risk_score']], on='Participant ID', how='left')
data.columns = [re.sub(r'[^\w_]', '_', col) if i in range(1, 171) else col 
                for i, col in enumerate(data.columns)]
data_phe = data
all_predictions = pd.read_csv(r"all_fold_predictions.csv")
merged_col = all_predictions['fold_0_pred']
for i in range(0, 10):
    merged_col = merged_col.combine_first(all_predictions[f'fold_{i}_pred'])
all_predictions['phe_pred'] = merged_col
all_predictions
data_phe = pd.merge(data_phe, all_predictions[['Participant ID', 'phe_pred']], on='Participant ID', how='left')
data_phe[['Participant ID', 'Region_Code','Sex','Ethnic_background','Body_mass_index__BMI_',
'Townsend_deprivation_index_at_recruitment','Age_at_recruitment','Ever_smoked']].to_csv('cov_data.csv', index=False)

protein相关性-cs

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import rankdata, norm
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.stats.multitest import multipletests
from joblib import Parallel, delayed
import warnings
import os
import warnings
import os
warnings.filterwarnings('ignore')

def getsampleIntersection(x, y, c, field, exclude_patients=False, exclude_file=None):
    """
    获取三个数据集的样本交集
    """
    if not exclude_patients:
        # 获取三个数据集的字段交集
        temp = set(x[field]).intersection(set(y[field]))
        intersection_field = list(temp.intersection(set(c[field])))
        
        # 设置索引并移除字段列
        x_intersect = x.set_index(field).loc[intersection_field]
        y_intersect = y.set_index(field).loc[intersection_field]
        c_intersect = c.set_index(field).loc[intersection_field]
        
    else:
        temp = set(x[field]).intersection(set(y[field]))
        intersection_field = list(temp.intersection(set(c[field])))
        
        # 排除特定患者
        if exclude_file and os.path.exists(exclude_file):
            patients = pd.read_csv(exclude_file, header=None)
            intersection_field = list(set(intersection_field) - set(patients[0]))
        
        x_intersect = x.set_index(field).loc[intersection_field]
        y_intersect = y.set_index(field).loc[intersection_field]
        c_intersect = c.set_index(field).loc[intersection_field]
    
    # 返回数据和列名
    x_columns = x_intersect.columns.tolist()
    y_columns = y_intersect.columns.tolist()
    c_columns = c_intersect.columns.tolist()
    
    return (x_intersect.values, y_intersect.values, c_intersect.values, 
            intersection_field, x_columns, y_columns, c_columns)

def inverseNorm_transformation(data):
    """
    逆正态变换
    """
    def rank_norm(x):
        # 处理缺失值
        valid_mask = ~np.isnan(x)
        x_clean = x[valid_mask]
        
        if len(x_clean) < 2:
            return np.full_like(x, np.nan)
        
        ranks = rankdata(x_clean, method='average')
        transformed = norm.ppf(ranks / (len(ranks) + 1))
        
        # 将结果放回原位置
        result = np.full_like(x, np.nan)
        result[valid_mask] = transformed
        return result
    
    return np.apply_along_axis(rank_norm, 0, data)
def save_results_to_csv(results_beta, results_se, results_tvalue, results_pvalue, 
                       padj_BH_overall, protein_names, region_names, output_file):
    """
    将结果保存为CSV文件
    """
    print("正在保存结果到CSV文件...")
    
    # 创建结果数据框
    results_list = []
    
    for i, protein in enumerate(protein_names):
        for j, region in enumerate(region_names):
            beta = results_beta[i, j]
            se = results_se[i, j]
            tval = results_tvalue[i, j]
            pval = results_pvalue[i, j]
            padj = padj_BH_overall[i, j]
            
            # 只保存有效结果
            if not np.isnan(beta) and not np.isnan(pval):
                results_list.append({
                    'Protein': protein,
                    'Brain_Region': region,
                    'Beta': beta,
                    'SE': se,
                    'T_value': tval,
                    'P_value': pval,
                    'P_adj_BH': padj,
                    'Significant_BH': padj < 0.05
                })
    
    if not results_list:
        print("警告: 没有有效的结果可保存")
        return pd.DataFrame()
    
    results_df = pd.DataFrame(results_list)
    
    # 按p值排序
    results_df = results_df.sort_values('P_value')
    
    # 保存到CSV
    results_df.to_csv(output_file, index=False)
    
    # 统计显著结果
    n_significant = np.sum(results_df['Significant_BH'])
    print(f"总关联数: {len(results_df)}")
    print(f"FDR校正后显著关联数: {n_significant}")
    
    if n_significant > 0:
        print("\n前10个最显著的关联:")
        print(results_df.head(10)[['Protein', 'Brain_Region', 'Beta', 'P_value', 'P_adj_BH']])
    
    return results_df

def ensure_float64(*arrays):
    """确保所有输入数组都是float64类型"""
    result = []
    for arr in arrays:
        if arr.dtype != np.float64:
            arr = arr.astype(np.float64)
        result.append(arr)
    return result if len(result) > 1 else result[0]

def batch_regression_fast(X, Y, C):
    """
    使用矩阵运算批量计算回归，修复数据类型问题
    """
    # 确保数据类型正确
    X, Y, C = ensure_float64(X, Y, C)
    
    n_samples, n_proteins = X.shape
    n_regions = Y.shape[1]
    
    # 准备设计矩阵（包括截距项）
    C_matrix = np.column_stack([np.ones(n_samples, dtype=np.float64), C])
    n_covariates = C_matrix.shape[1]
    
    # 预分配结果数组
    results_beta = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_se = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_tvalue = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_pvalue = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    
    for i in range(n_proteins):
        if i % 100 == 0:
            print(f"处理蛋白质 {i}/{n_proteins}")
            
        # 当前蛋白质数据
        x = X[:, i]
        valid_mask = ~np.isnan(x)
        
        if np.sum(valid_mask) < 10:
            continue
            
        x_valid = x[valid_mask]
        X_design_valid = np.column_stack([
            C_matrix[valid_mask], 
            x_valid.reshape(-1, 1)
        ]).astype(np.float64)
        
        # 一次性计算所有脑区的回归
        for j in range(n_regions):
            y = Y[:, j]
            y_valid_mask = valid_mask & ~np.isnan(y)
            
            if np.sum(y_valid_mask) < 10:
                continue
                
            y_valid = y[y_valid_mask]
            X_design_current = X_design_valid[y_valid_mask[valid_mask]]
            
            try:
                # 更稳定的矩阵求逆方法
                XTX = X_design_current.T @ X_design_current
                
                # 检查矩阵条件数，避免奇异矩阵
                cond_num = np.linalg.cond(XTX)
                if cond_num > 1e10:  # 条件数过大，矩阵接近奇异
                    continue
                
                # 使用更稳定的求逆方法
                try:
                    XTX_inv = np.linalg.inv(XTX)
                except np.linalg.LinAlgError:
                    # 如果直接求逆失败，使用伪逆
                    XTX_inv = np.linalg.pinv(XTX)
                
                coefficients = XTX_inv @ X_design_current.T @ y_valid
                
                # 计算统计量
                y_pred = X_design_current @ coefficients
                residuals = y_valid - y_pred
                n_valid = len(y_valid)
                p = X_design_current.shape[1]
                
                if n_valid <= p:
                    continue
                    
                mse = np.sum(residuals**2) / (n_valid - p)
                
                # 蛋白质系数的标准误（最后一个系数）
                se = np.sqrt(mse * XTX_inv[-1, -1])
                t_value = coefficients[-1] / se
                
                # 更安全的p值计算
                if np.isfinite(t_value):
                    p_value = 2 * (1 - stats.t.cdf(np.abs(t_value), n_valid - p))
                else:
                    p_value = np.nan
                
                results_beta[i, j] = coefficients[-1]
                results_se[i, j] = se
                results_tvalue[i, j] = t_value
                results_pvalue[i, j] = p_value
                
            except Exception as e:
                continue
    
    return results_beta, results_se, results_tvalue, results_pvalue

def process_chunk_stable(args):
    """
    更稳定的分块处理函数
    """
    chunk_indices, X_chunk, Y, C_matrix = args
    n_regions = Y.shape[1]
    
    # 确保数据类型
    X_chunk, Y, C_matrix = ensure_float64(X_chunk, Y, C_matrix)
    
    chunk_results = []
    
    for idx_in_chunk, i in enumerate(chunk_indices):
        x = X_chunk[:, idx_in_chunk]
        valid_mask = ~np.isnan(x)
        
        if np.sum(valid_mask) < 10:
            chunk_results.append((i, np.full((n_regions, 4), np.nan, dtype=np.float64)))
            continue
            
        x_valid = x[valid_mask]
        X_design_valid = np.column_stack([
            C_matrix[valid_mask], 
            x_valid.reshape(-1, 1)
        ]).astype(np.float64)
        
        protein_results = []
        for j in range(n_regions):
            y = Y[:, j]
            y_valid_mask = valid_mask & ~np.isnan(y)
            
            if np.sum(y_valid_mask) < 10:
                protein_results.extend([np.nan] * 4)
                continue
                
            try:
                y_valid = y[y_valid_mask].astype(np.float64)
                X_design_current = X_design_valid[y_valid_mask[valid_mask]]
                
                # 稳定的矩阵运算
                XTX = X_design_current.T @ X_design_current
                cond_num = np.linalg.cond(XTX)
                
                if cond_num > 1e10:
                    protein_results.extend([np.nan] * 4)
                    continue
                
                try:
                    XTX_inv = np.linalg.inv(XTX)
                except np.linalg.LinAlgError:
                    XTX_inv = np.linalg.pinv(XTX)
                
                coefficients = XTX_inv @ X_design_current.T @ y_valid
                
                y_pred = X_design_current @ coefficients
                residuals = y_valid - y_pred
                n_valid = len(y_valid)
                p = X_design_current.shape[1]
                
                if n_valid <= p:
                    protein_results.extend([np.nan] * 4)
                    continue
                
                mse = np.sum(residuals**2) / (n_valid - p)
                se = np.sqrt(mse * XTX_inv[-1, -1])
                t_value = coefficients[-1] / se
                
                if np.isfinite(t_value):
                    p_value = 2 * (1 - stats.t.cdf(np.abs(t_value), n_valid - p))
                else:
                    p_value = np.nan
                
                protein_results.extend([coefficients[-1], se, t_value, p_value])
                
            except Exception as e:
                protein_results.extend([np.nan] * 4)
        
        chunk_results.append((i, np.array(protein_results, dtype=np.float64).reshape(-1, 4)))
    
    return chunk_results

def parallel_regression_stable(X, Y, C, n_jobs=20, chunk_size=50):
    """
    稳定的并行回归分析
    """
    n_samples, n_proteins = X.shape
    n_regions = Y.shape[1]
    
    # 准备设计矩阵并确保数据类型
    C_matrix = np.column_stack([np.ones(n_samples, dtype=np.float64), C])
    C_matrix = ensure_float64(C_matrix)
    
    # 分块处理蛋白质
    protein_indices = list(range(n_proteins))
    chunks = []
    
    for i in range(0, n_proteins, chunk_size):
        chunk_indices = protein_indices[i:i + chunk_size]
        X_chunk = X[:, chunk_indices]
        chunks.append((chunk_indices, X_chunk, Y, C_matrix))
    
    # 并行处理
    print(f"使用 {n_jobs} 个进程并行处理 {len(chunks)} 个数据块...")
    chunk_results = Parallel(n_jobs=n_jobs)(
        delayed(process_chunk_stable)(chunk) for chunk in chunks
    )
    
    # 合并结果
    results_beta = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_se = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_tvalue = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    results_pvalue = np.full((n_proteins, n_regions), np.nan, dtype=np.float64)
    
    for chunk in chunk_results:
        for protein_idx, protein_result in chunk:
            if not np.all(np.isnan(protein_result)):
                results_beta[protein_idx] = protein_result[:, 0]
                results_se[protein_idx] = protein_result[:, 1]
                results_tvalue[protein_idx] = protein_result[:, 2]
                results_pvalue[protein_idx] = protein_result[:, 3]
    
    return results_beta, results_se, results_tvalue, results_pvalue

# 在主流程中添加数据预处理步骤
def preprocess_and_ensure_numeric(X_vals, Y_vals, C_vals):
    """
    数据预处理：确保所有数据都是数值类型
    """
    print("确保数据为数值类型...")
    
    # 处理X数据
    if hasattr(X_vals, 'dtype') and X_vals.dtype == object:
        X_vals = X_vals.astype(np.float64)
    
    # 处理Y数据
    if hasattr(Y_vals, 'dtype') and Y_vals.dtype == object:
        Y_vals = Y_vals.astype(np.float64)
    
    # 处理C数据
    if hasattr(C_vals, 'dtype') and C_vals.dtype == object:
        C_vals = C_vals.astype(np.float64)
    
    # 替换无穷值为NaN
    X_vals = np.where(np.isfinite(X_vals), X_vals, np.nan)
    Y_vals = np.where(np.isfinite(Y_vals), Y_vals, np.nan)
    C_vals = np.where(np.isfinite(C_vals), C_vals, np.nan)
    
    print(f"数据类型: X={X_vals.dtype}, Y={Y_vals.dtype}, C={C_vals.dtype}")
    
    return X_vals, Y_vals, C_vals

In [ ]:
# 1. 读取数据
print("1. 读取数据...")
X = data_protein.rename(columns={'Participant ID':'eid'}).drop(columns=['depressed-after','Region_Code'])
Y = df_cs.rename(columns={'participant.eid':'eid'})

# 2. 读取协变量并处理
print("2. 处理协变量数据...")

# 选择需要的列
cov = co.rename(columns={'Participant ID': 'eid',
                                        'Body_mass_index__BMI_': 'BMI',
                                        'Townsend_deprivation_index_at_recruitment': 'Townsend_index',
                                        'Age_at_recruitment': 'Age',
                                        'Ever_smoked': 'Smoking'})
# 创建哑变量
cov = pd.get_dummies(cov, columns=['Region_Code', 'Ethnic_background'], drop_first=True)

# 3. 获取样本交集
print("3. 获取样本交集...")
X_vals, Y_vals, C_vals, intersection_field, x_cols, y_cols, c_cols = getsampleIntersection(
    X, Y, cov, 'eid', True
)

print(f"最终分析样本数: {len(intersection_field)}")
print(f"蛋白质数据形状: {X_vals.shape}")
print(f"脑体积数据形状: {Y_vals.shape}")
print(f"协变量数据形状: {C_vals.shape}")
print(f"协变量列数: {len(c_cols)}")

# 4. 逆正态变换
print("4. 对脑体积数据进行逆正态变换...")
Y_vals = inverseNorm_transformation(Y_vals)

# 5. 转换为DataFrame便于建模
C_df = pd.DataFrame(C_vals, columns=c_cols)
print(f"C_df形状: {C_df.shape}")

# 6. 读取蛋白质和脑区名称
protein_names = data_protein.columns.tolist()[3:]  # 排除'eid'列

region_names = df_cs.columns.tolist()[1:]  # 排除'eid'列

# 7. 并行执行回归分析 - 使用所有蛋白质和脑区
print("5. 执行回归分析...")
n_proteins = X_vals.shape[1]
n_regions = Y_vals.shape[1]

print(f"分析规模: {n_proteins}个蛋白质 × {n_regions}个脑区 = {n_proteins * n_regions}个关联")
# 创建所有组合 - 使用所有蛋白质和脑区
combinations = [(i, j) for i in range(n_proteins) for j in range(n_regions)]

print(f"实际分析: {n_proteins}个蛋白质 × {n_regions}个脑区 = {len(combinations)}个关联")

# 数据预处理
X_vals, Y_vals, C_vals = preprocess_and_ensure_numeric(X_vals, Y_vals, C_vals)

# 选择分析方法
if n_proteins * n_regions > 100000:
    print("使用并行稳定版本处理大数据集...")
    results_beta, results_se, results_tvalue, results_pvalue = parallel_regression_stable(
        X_vals, Y_vals, C_vals, n_jobs=60, chunk_size=20
    )
else:
    print("使用批量稳定版本...")
    results_beta, results_se, results_tvalue, results_pvalue = batch_regression_fast(
        X_vals, Y_vals, C_vals
    )


# 10. 多重检验校正 - FDR校正
print("7. 多重检验校正 (FDR)...")
pvalues_flat = results_pvalue.flatten()
valid_pvalues = pvalues_flat[~np.isnan(pvalues_flat)]

if len(valid_pvalues) > 0:
    # 使用FDR校正 (Benjamini-Hochberg)
    _, padj_BH_overall_flat, _, _ = multipletests(valid_pvalues, method='fdr_bh')
    
    # 重构校正后的p值矩阵
    padj_BH_overall = np.full_like(results_pvalue, np.nan)
    valid_mask = ~np.isnan(results_pvalue)
    padj_BH_overall[valid_mask] = padj_BH_overall_flat
else:
    padj_BH_overall = np.full_like(results_pvalue, np.nan)
    print("警告: 没有有效的p值进行多重检验校正")

# 11. 保存结果到CSV
print("8. 保存结果...")
results_df = save_results_to_csv(
    results_beta, results_se, results_tvalue, results_pvalue,
    padj_BH_overall, 
    protein_names,  # 使用所有蛋白质
    region_names,   # 使用所有脑区
    'protein_cs_association_results.csv'
)

print("\n分析完成！")
print(f"结果已保存到: protein_cs_association_results.csv")

# 显示一些统计信息
if not results_df.empty:
    print(f"\n结果统计:")
    print(f"- 总关联数: {len(results_df)}")
    print(f"- 显著关联数 (FDR < 0.05): {sum(results_df['Significant_BH'])}")
    print(f"- 最显著的关联: {results_df.iloc[0]['Protein']} - {results_df.iloc[0]['Brain_Region']} (p = {results_df.iloc[0]['P_value']:.6f})")
    
    # 按蛋白质和脑区统计显著关联
    if sum(results_df['Significant_BH']) > 0:
        sig_results = results_df[results_df['Significant_BH']]
        print(f"\n显著关联按蛋白质分布:")
        protein_sig_counts = sig_results['Protein'].value_counts()
        print(protein_sig_counts.head(10))
        
        print(f"\n显著关联按脑区分布:")
        region_sig_counts = sig_results['Brain_Region'].value_counts()
        print(region_sig_counts.head(10))

## 数据处理

In [ ]:
import pandas as pd
pc = pd.read_csv(r"protein_cs_association_results.csv")
ps = pd.read_csv(r"protein_ss_association_results.csv")
pg = pd.read_csv(r"protein_gv_association_results.csv")
bc = pd.read_csv(r"bio_cs_association_results.csv")
bs = pd.read_csv(r"bio_ss_association_results.csv")
bg = pd.read_csv(r"bio_gv_association_results.csv")

cs

In [ ]:
import pandas as pd
# 假定 pbc 已在当前环境中
# 过滤显著项
pbc = pd.concat([pc, bc],  axis=0)
sig = pbc[pbc['P_adj_BH'] < 0.01].copy()
# 统计每个 Brain_Region 的显著 Protein 数（去重 Protein）
counts = sig.groupby('Brain_Region')['Protein'].nunique().sort_values(ascending=False)

# 将结果转为 DataFrame 并保存
counts_df = counts.reset_index().rename(columns={'Protein': 'n_significant_proteins'})
out_path = r"pbc_significant_counts_by_region.csv"
id = pd.read_csv(r"mental_mri\id\cobrainid.csv")
df_cs = pd.merge(id,counts_df,left_on='id', right_on='Brain_Region', how='left')
df_cs['n_significant_proteins'].fillna(0, inplace=True)
df_cs

In [ ]:
# ...existing code...
# 按 type 列分为最多 4 个 DataFrame 并保存
if 'type' not in df_cs.columns:
    raise KeyError("DataFrame 中不存在 'type' 列。")

unique_types = df_cs['type'].dropna().unique().tolist()
print(f"发现的唯一 type 值: {unique_types}")
if len(unique_types) == 0:
    raise ValueError("type 列中没有有效值。")

data_by_type = {t: df_cs[df_cs['type'] == t].copy() for t in unique_types}
data_by_type['Area'] #Thickness，Volume

In [ ]:
# ...existing code...
# 按 type 列分为最多 4 个 DataFrame 并保存
if 'type' not in df_cs.columns:
    raise KeyError("DataFrame 中不存在 'type' 列。")

unique_types = df_cs['type'].dropna().unique().tolist()
print(f"发现的唯一 type 值: {unique_types}")
if len(unique_types) == 0:
    raise ValueError("type 列中没有有效值。")

data_by_type = {t: df_cs[df_cs['type'] == t].copy() for t in unique_types}
data_by_type['Thickness']['n_significant_proteins'].tolist()

ss

In [ ]:
import pandas as pd
# 假定 pbc 已在当前环境中
# 过滤显著项
pbs = pd.concat([ps, bs],  axis=0)
sig = pbs[pbs['P_adj_BH'] < 0.01].copy()
# 统计每个 Brain_Region 的显著 Protein 数（去重 Protein）
counts = sig.groupby('Brain_Region')['Protein'].nunique().sort_values(ascending=False)

# 将结果转为 DataFrame 并保存
counts_df = counts.reset_index().rename(columns={'Protein': 'n_significant_proteins'})
out_path = r"pbc_significant_counts_by_region.csv"
id = pd.read_csv(r"mental_mri\id\cobrainid - 副本.csv")
df_ss = pd.merge(id,counts_df,left_on='id', right_on='Brain_Region', how='left')
df_ss['n_significant_proteins'].fillna(0, inplace=True)
df_ss

In [ ]:
# ...existing code...
# 按 type 列分为最多 4 个 DataFrame 并保存
if 'type' not in df_ss.columns:
    raise KeyError("DataFrame 中不存在 'type' 列。")

unique_types = df_ss['type'].dropna().unique().tolist()
print(f"发现的唯一 type 值: {unique_types}")
if len(unique_types) == 0:
    raise ValueError("type 列中没有有效值。")

data_by_type = {t: df_ss[df_ss['type'] == t].copy() for t in unique_types}
data_by_type['Volume']['n_significant_proteins'].tolist()

gv

In [ ]:
bio = pd.read_csv(r"Gwas\PHESANT-master\resultsallbio1\merged.csv")
bio_select = bio[bio['bonferroni']<0.01]
id = pd.read_csv(r'Gwas\PHESANT-master\2depress_biowas\id.csv')
bio_select = pd.merge(bio_select[['varName_x','Cat2_Title']],id,left_on='varName_x',right_on='UKB Field ID',how='left')
bio_select

In [ ]:
import pandas as pd
pbg = pd.concat([pg, bg],  axis=0)
pbg = pd.merge(pbg, bio_select[['Biomarker','Cat2_Title']], left_on='Protein', right_on='Biomarker', how='left')
pbg['Cat2_Title'].fillna('Protein', inplace=True)
pbg.drop(columns='Biomarker',inplace = True)

In [ ]:
if 'Brain_Region' not in pbg.columns:
    raise KeyError("pbg 中不存在 'Brain_Region' 列，请确认源数据列名。")

def _map_region(val):
    if pd.isna(val):
        return val
    s = str(val)
    if '26517' in s:
        return 'SGV'
    if '26518' in s:
        return 'TGV'
    if '26528' in s:
        return 'WMH'
    return val

pbg['Brain_Region'] = pbg['Brain_Region'].apply(_map_region)
pbg

In [ ]:
pbg.to_csv('../result/protein_bio_gv_association_results1.csv')